# 5 - Ticket suggestion: retrieve the cases already solved this way

**Question.** A new ticket arrives. Put the K resolved tickets closest to it in
front of the agent, with what was done, before they start from scratch. Which
retriever finds them?

| Approach | What it is |
|---|---|
| `lexical` | BM25 over ticket text |
| `dense_tfidf` | Cosine over TF-IDF vectors |
| `dense_embed` | Cosine over multilingual sentence embeddings |
| `hybrid` | Reciprocal-rank fusion of lexical + dense |

**Every dependency is mandatory** - the embedding arm used to be skippable, so
the hybrid could silently degrade to TF-IDF cosine. All four arms run.

### Read this before the numbers

**The relevance labels are a proxy.** Nobody has judged which resolved ticket
is genuinely useful for a new one, so a retrieved case counts as relevant when
it shares the new ticket's `category` + `subcategory`. That is standard when no
judgements exist, and it is **weaker than it looks**: it rewards a retriever
for finding topically similar tickets, which is not the same as finding ones
whose *resolution* transfers. Every number below inherits that. Section 6 is
where a human has to look.

**No vector database.** At this corpus size an in-memory matrix and a dot
product is the same maths a vector store runs, minus the deployment. If hybrid
retrieval works here, *choosing a store* is the next question and a different
one.

**Fusion is on ranks, not scores.** BM25 is unbounded, cosine is in [-1, 1].
Normalising them onto a common scale means inventing a mapping that changes
with every query. RRF only uses position.

```bash
uv sync
uv run itsm data build
```

### Runtime

BM25 is precomputed into per-term contribution arrays at index time, so scoring
a query is a handful of numpy scatter-adds rather than a Python loop over the
corpus - the difference between minutes and seconds at 300 queries. The dense
arm encodes 3000 documents once (~30 s on CPU, cached encoder) and every query
batch is then a single matrix product.

In [ ]:
import json
import math
import re
import time
from collections import Counter

import numpy as np
import pandas as pd

from itsm.config import get_settings
from itsm.pipelines.anonymise import load_raw_tickets
from itsm.text.vectorize import build_vectorizer

pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 80)

settings = get_settings()
TEXT_COLUMN = 'designation'
N_CORPUS = 3000     # resolved tickets indexed
N_QUERIES = 300     # new tickets searched with
K = 5               # what the agent actually sees
DEPTH = 50          # candidates per arm before fusion
RRF_K = 60          # rank damping; 60 is the convention from the RRF paper

## 1. Corpus and queries - split in time

The corpus is **resolved tickets from the past**; the queries are **later
tickets**. A random split would let a retriever find a near-duplicate filed the
same afternoon, which is not the situation this feature is for.

The first thing to check is whether resolutions exist at all. Retrieval and
"suggest a mitigation" both die without usable resolution text, and in most
real ITSM exports close notes are empty or boilerplate. **If the coverage
printed below is low on your own extract, stop here** - that is the finding.

In [ ]:
tickets = load_raw_tickets(settings).sort_values('opened_at').reset_index(drop=True)

resolution_columns = [c for c in ('close_notes', 'resolution', 'work_notes') if c in tickets.columns]
for column in resolution_columns:
    filled = tickets[column].fillna('').astype(str).str.strip().str.len() > 15
    print(f'{column}: {filled.mean():.1%} of tickets carry usable text')
if not resolution_columns:
    print('WARNING: no resolution column in this extract. Retrieval can rank '
          'similar tickets, but there is nothing to suggest back to the agent.')

cut = len(tickets) - N_QUERIES
corpus_frame = tickets.iloc[max(0, cut - N_CORPUS):cut].reset_index(drop=True)
query_frame = tickets.iloc[cut:].reset_index(drop=True)

corpus_texts = corpus_frame[TEXT_COLUMN].fillna('').astype(str).tolist()
query_texts = query_frame[TEXT_COLUMN].fillna('').astype(str).tolist()
print(f'\n{len(corpus_texts)} indexed / {len(query_texts)} queries')

In [ ]:
# The proxy relevance set: same category + subcategory. Stated as a key, so the
# assumption is visible in the code rather than buried in a docstring.
def relevance_key(frame):
    parts = [frame['category'].astype(str)]
    if 'subcategory' in frame.columns:
        parts.append(frame['subcategory'].astype(str))
    return parts[0].str.cat(parts[1:], sep=' | ') if len(parts) > 1 else parts[0]


corpus_keys = relevance_key(corpus_frame).tolist()
query_keys = relevance_key(query_frame).tolist()

by_key = {}
for position, key in enumerate(corpus_keys):
    by_key.setdefault(key, set()).add(position)

relevant = [by_key.get(key, set()) for key in query_keys]
answerable = [i for i, gold in enumerate(relevant) if gold]
print(f'{len(answerable)}/{len(relevant)} queries have at least one relevant case in the corpus')
print('mean relevant per query:', round(np.mean([len(r) for r in relevant]), 1))

## 2. The retrievers

BM25 is written out rather than pulled in for one reason: the whole comparison
rests on lexical retrieval finding the tokens that *identify* an ITSM ticket -
`ORA-01017`, `SRV-PARIS-07`, `0x80070005`. A library's default tokeniser splits
on non-alphanumerics and turns `ORA-01017` into `ora` and `01017`, destroying
exactly what is being tested. The tokeniser is the substantive choice here, so
it is visible.

In [ ]:
# Keeps internal hyphens, dots and digits together: error codes, hostnames and
# versions survive as single terms.
TOKEN = re.compile(r'[A-Za-z0-9\u00C0-\u024F]+(?:[-._/][A-Za-z0-9]+)*')


def tokenize(text):
    return [m.group(0).lower() for m in TOKEN.finditer(text or '')]


print(tokenize('Erreur ORA-01017 sur SRV-PARIS-07 apres la maj v2.1'))


class BM25:
    '''Okapi BM25. k1 = 1.5, b = 0.75, the standard settings.

    The per-(term, document) contribution depends only on the term frequency
    and the document length, never on the query - so all of it is computed once
    at index time into two flat arrays per term. Scoring a query is then a
    scatter-add per query term instead of a Python loop over the corpus.
    '''

    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b

    def index(self, texts):
        docs = [Counter(tokenize(t)) for t in texts]
        self.n_docs = len(docs)
        lengths = np.array([sum(d.values()) for d in docs], dtype=float)
        avgdl = float(lengths.mean()) if self.n_docs else 0.0
        norm = 1 - self.b + self.b * lengths / (avgdl or 1.0)

        frequency = Counter()
        for doc in docs:
            frequency.update(doc.keys())

        # term -> (document indices, precomputed contributions)
        self.postings = {}
        raw = {}
        for position, doc in enumerate(docs):
            for term, tf in doc.items():
                raw.setdefault(term, []).append((position, tf))
        for term, entries in raw.items():
            idf = math.log(1 + (self.n_docs - frequency[term] + 0.5) / (frequency[term] + 0.5))
            indices = np.fromiter((p for p, _ in entries), dtype=np.int32, count=len(entries))
            tfs = np.fromiter((t for _, t in entries), dtype=float, count=len(entries))
            weights = idf * tfs * (self.k1 + 1) / (tfs + self.k1 * norm[indices])
            self.postings[term] = (indices, weights)
        return self

    def search(self, queries, k=10):
        out = []
        scores = np.empty(self.n_docs)
        for query in queries:
            scores.fill(0.0)
            for term in set(tokenize(query)):
                posting = self.postings.get(term)
                if posting is not None:
                    np.add.at(scores, posting[0], posting[1])
            out.append(np.argsort(-scores)[:k].tolist())
        return out

In [ ]:
class Dense:
    '''Cosine similarity over a shared vector space.

    `tfidf` is not dense in any meaningful sense - it is sparse and lexical -
    but including it answers a question worth asking: how much of the embedding
    arm's advantage is semantics, and how much is just cosine over a different
    weighting than BM25 uses?

    All queries are encoded and scored in one matrix product; the encoder
    itself is cached process-wide by `itsm.text.vectorize`.
    '''

    def __init__(self, backend='embeddings'):
        self.backend = backend

    @staticmethod
    def _unit(matrix):
        dense = np.asarray(matrix.todense()) if hasattr(matrix, 'todense') else np.asarray(matrix)
        norms = np.linalg.norm(dense, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return dense / norms

    def index(self, texts):
        self.vectorizer = build_vectorizer(self.backend, settings.embedding_model)
        self.matrix = self._unit(self.vectorizer.fit_transform(texts))
        return self

    def search(self, queries, k=10):
        similarity = self._unit(self.vectorizer.transform(list(queries))) @ self.matrix.T
        return [np.argsort(-row)[:k].tolist() for row in similarity]


def rrf(rankings, weights=None, k=RRF_K, top_k=K):
    '''Reciprocal-rank fusion: sum of weight / (k + rank) over retrievers.'''
    weights = weights or [1.0] * len(rankings)
    fused = {}
    for ranking, weight in zip(rankings, weights):
        if not weight:
            continue
        for rank, doc in enumerate(ranking, start=1):
            fused[doc] = fused.get(doc, 0.0) + weight / (k + rank)
    return [doc for doc, _ in sorted(fused.items(), key=lambda item: -item[1])[:top_k]]

## 3. Run

Each arm retrieves `DEPTH` candidates so fusion has something to work with: a
document ranked 30th by one arm and 2nd by the other is exactly the case hybrid
exists for, and a shallow cut hides it.

In [ ]:
rankings, timings = {}, {}

for name, retriever in (('lexical', BM25()), ('dense_tfidf', Dense('tfidf')),
                        ('dense_embed', Dense('embeddings'))):
    started = time.perf_counter()
    retriever.index(corpus_texts)
    index_seconds = time.perf_counter() - started

    started = time.perf_counter()
    rankings[name] = retriever.search(query_texts, k=DEPTH)
    timings[name] = time.perf_counter() - started
    print(f'{name:12s} index {index_seconds:6.1f}s  search {timings[name]:6.1f}s'
          f'  {1000 * timings[name] / len(query_texts):6.1f} ms/query')

started = time.perf_counter()
rankings['hybrid'] = [
    rrf([rankings['lexical'][i], rankings['dense_embed'][i]], top_k=DEPTH)
    for i in range(len(query_texts))
]
timings['hybrid'] = timings['lexical'] + timings['dense_embed'] + (time.perf_counter() - started)
print('hybrid: lexical + dense_embed')

In [ ]:
def evaluate(ranked, gold_sets, k_values=(1, 3, 5, 10)):
    out = {}
    for k in k_values:
        recalls, hits_at_k = [], []
        for ranking, gold in zip(ranked, gold_sets):
            if not gold:
                continue
            hits = len(set(ranking[:k]) & gold)
            recalls.append(hits / len(gold))
            hits_at_k.append(1.0 if hits else 0.0)
        out['recall@' + str(k)] = round(float(np.mean(recalls)), 4)
        # Hit rate is the number the agent feels: did the panel contain
        # anything useful at all? Recall@k over a large relevant set does not
        # say that.
        out['hit rate@' + str(k)] = round(float(np.mean(hits_at_k)), 4)
    reciprocal = []
    for ranking, gold in zip(ranked, gold_sets):
        if not gold:
            continue
        rank = next((i + 1 for i, doc in enumerate(ranking) if doc in gold), 0)
        reciprocal.append(1 / rank if rank else 0.0)
    out['MRR'] = round(float(np.mean(reciprocal)), 4)
    return out


scores = {}
for name, ranked in rankings.items():
    scores[name] = evaluate(ranked, relevant)
    scores[name]['ms per query'] = round(1000 * timings[name] / len(query_texts), 2)

comparison = pd.DataFrame(scores)
comparison

## 4. Does the hybrid earn its complexity?

Hybrid costs two indexes, two runtimes and a fusion parameter. It has to buy
something for that. The sweep below runs the weight from pure lexical to pure
dense - if the curve is flat, or if an endpoint wins, **the hybrid is not the
answer here** and one arm ships alone.

In [ ]:
sweep = []
for weight in (0.0, 0.25, 0.5, 0.75, 1.0):
    fused = [
        rrf([rankings['lexical'][i], rankings['dense_embed'][i]],
            weights=[weight, 1 - weight], top_k=10)
        for i in range(len(query_texts))
    ]
    row = {'lexical weight': weight}
    row.update(evaluate(fused, relevant, k_values=(5,)))
    sweep.append(row)

pd.DataFrame(sweep).set_index('lexical weight')

## 5. Where the arms disagree

The queries where lexical wins and dense loses are the error-code cases; the
reverse are the paraphrases. If both sets are large, the hybrid has a real job.
If one is empty, it does not.

In [ ]:
lexical_only, dense_only = [], []
for i in answerable:
    gold = relevant[i]
    lex_hit = bool(set(rankings['lexical'][i][:K]) & gold)
    dense_hit = bool(set(rankings['dense_embed'][i][:K]) & gold)
    if lex_hit and not dense_hit:
        lexical_only.append(i)
    elif dense_hit and not lex_hit:
        dense_only.append(i)

print(f'lexical finds it, dense does not: {len(lexical_only)} queries')
for i in lexical_only[:5]:
    print('   ', query_texts[i][:100])
print(f'\ndense finds it, lexical does not: {len(dense_only)} queries')
for i in dense_only[:5]:
    print('   ', query_texts[i][:100])

## 6. What the agent would actually see

**Read this before believing the table.** The metrics score agreement with a
category proxy; this cell shows the real deliverable. For each of a handful of
queries: the new ticket, and the K cases that would be suggested.

The question to ask of each panel is not *is it the same category* but **would
this have saved the agent time**. Twenty of these read by hand is worth more
than another decimal place of recall@5.

In [ ]:
best = max(scores, key=lambda name: scores[name]['hit rate@5'])
print('best arm by hit rate@5:', best, '\n')

columns = [TEXT_COLUMN, 'category'] + resolution_columns[:1]
for i in answerable[:4]:
    print('QUERY:', query_texts[i])
    print('  true category:', query_keys[i])
    panel = corpus_frame.iloc[rankings[best][i][:K]][columns].copy()
    panel.insert(0, 'relevant?', [pos in relevant[i] for pos in rankings[best][i][:K]])
    display(panel)
    print()

## 7. Conclusion

*Fill in from the run. The structure of the argument:*

**What ships.** The arm that wins `hit rate@5` at a latency inside the submit
budget - and if the hybrid does not beat both arms in section 4, it does not
ship: two indexes, two runtimes and a fusion weight are real cost, and
"everyone uses hybrid" is not a measurement.

**`hit rate@5` over `recall@5`.** The agent sees five cases. Whether the panel
contained anything useful is the experience; recall over a relevant set of 200
same-category tickets is not.

**The load-bearing question is upstream of all of this.** If the resolution
coverage printed in section 1 is low on the real extract, the best retriever in
this table still has nothing to suggest. Sample 200 closed tickets from
production and read the close notes before building any of it - a one-day check
that decides a quarter of work.

**Threats to validity:**

* **The relevance proxy.** Same category is not the same fix. Real judgements -
  even 50 queries with a human marking which suggestions helped - would replace
  every number in the headline, and should, before this ships.
* Generated designations are more templated than real ones, which flatters
  lexical retrieval.
* One embedding model, one tokeniser, one fusion constant. Each is a sweep
  nobody has run yet.
* No reranker. A cross-encoder over the top 50 is the obvious next step if the
  ceiling here is ranking rather than recall.

In [ ]:
settings.ensure_dirs()
path = settings.reports_dir / 'experiment_05_retrieval.json'
path.write_text(
    json.dumps(
        {
            'text_column': TEXT_COLUMN,
            'n_corpus': len(corpus_texts),
            'n_queries': len(query_texts),
            'relevance': 'proxy: same category + subcategory (not human judgements)',
            'k': K,
            'headline': comparison.to_dict(),
            'fusion_sweep': sweep,
        },
        indent=2,
        default=str,
    ),
    encoding='utf-8',
)
print('written to', path)